Импорт

In [21]:

import json
import requests
import time #для пауз
from typing import List, Dict
from cbr_data import Cbr

Получение одной страницы

In [31]:
def get_vacancies_page(
        text: str = "Python",
        area: str = "1",
        only_with_salary: bool = True,
        page: int = 1,
        per_page: int = 10,
) -> Dict:

    '''
    функция принимает:
    text - что ищем
    area - регион (1 - Москва, 113 - Россия)
    only_with_salary - c указанием зп/без
    per_page - максимум на страницу
    после присваивается ссылка на api.hh.ru и введённые параметры
    headers для обращения к API, чтобы сервер понимал, что это человек, а не ddos)))
    response - (с англ. "ответ") принимает итог после обращения к серверу
    условие, если не 200(т.е. неудачный итог), то выписать ошибку
    '''

    url = "https://api.hh.ru/vacancies"
    params = {
        "text": text,
        "area": area,
        "only_with_salary": str(only_with_salary).lower(),
        "per_page": per_page,
        "page": page
    }
    headers = {"User-Agent": "PersonalResearch/2026 (my.telegram @rizottonero)"}
    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        print(f"Error: {response.status_code} - {response.text}")
        return {}

    return response.json()



Сбор нескольких страниц

In [32]:
def collect_vacancies(
        text: str = 'Python',
        area: str = "1",
        count_pages: int = 5,
        sleep_sec: float = 1.5,
) -> List[Dict]:
    all_vacancies = []
    for page in range(count_pages):
        data = get_vacancies_page(text=text, area=area, page=page)
        if not data or "items" not in data:
            break

        all_vacancies.extend(data["items"])

        if page >= data.get("pages", 1) - 1:
            break

        time.sleep(sleep_sec)

    print(f"Собрано вакансий: {len(all_vacancies)}")
    return all_vacancies


Запуск кода

In [33]:
if __name__ == "__main__":
    vacancies = collect_vacancies(
        text="Data Scientist OR ML engineer",
        area="113",
        count_pages=10,
        sleep_sec=2.0
    )

    with open("vacancies.json", "w", encoding="utf-8") as f:
        json.dump(vacancies, f, ensure_ascii=False, indent=2)


Собрано вакансий: 100


In [34]:
if __name__ == "__main__":
    import pandas as pd
    df = pd.json_normalize(vacancies)  # распаковывает вложенные словари
    df.to_csv("vacancies.csv", index=False)
    print(df.head())

          id  premium                                              name  \
0  129611136    False  Senior Data Scientist – Renewable Energy Trading   
1  128433714    False          ML Engineer / Инженер машинного обучения   
2  129649717    False               Senior ML Engineer / Data Scientist   
3  130055660    False          Data Scientist (аудит/валидация моделей)   
4  129286350    False       Senior ML Engineer / Ведущий ML разработчик   

   department  has_test  response_letter_required  address response_url  \
0         NaN     False                      True      NaN         None   
1         NaN     False                     False      NaN         None   
2         NaN     False                     False      NaN         None   
3         NaN     False                     False      NaN         None   
4         NaN     False                     False      NaN         None   

  sort_point_distance              published_at  ...  \
0                None  2026-01-31T16:46:05

Осмотр данных

In [35]:
print(df.shape) #(количество строк, столбцов)
print(df.columns.tolist())
print(df.head(2))

(100, 99)
['id', 'premium', 'name', 'department', 'has_test', 'response_letter_required', 'address', 'response_url', 'sort_point_distance', 'published_at', 'created_at', 'archived', 'apply_alternate_url', 'show_logo_in_search', 'show_contacts', 'insider_interview', 'url', 'alternate_url', 'relations', 'contacts', 'working_days', 'working_time_intervals', 'working_time_modes', 'accept_temporary', 'fly_in_fly_out_duration', 'work_format', 'working_hours', 'work_schedule_by_days', 'accept_labor_contract', 'civil_law_contracts', 'night_shifts', 'professional_roles', 'accept_incomplete_resumes', 'internship', 'adv_response_url', 'is_adv_vacancy', 'adv_context', 'area.id', 'area.name', 'area.url', 'salary.from', 'salary.to', 'salary.currency', 'salary.gross', 'salary_range.from', 'salary_range.to', 'salary_range.currency', 'salary_range.gross', 'salary_range.mode.id', 'salary_range.mode.name', 'salary_range.frequency.id', 'salary_range.frequency.name', 'type.id', 'type.name', 'employer.id', 

In [41]:
df.info() # типы данных
df.isnull().sum() # сумма пропусков
print("Вакансий без зарплаты:", df['salary.from'].isnull().sum() + df['salary.to'].isnull().sum())
print(df['area.name'].value_counts().head(10))
print(df['experience.name'].value_counts())
df.describe() # статистика

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 99 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   id                               100 non-null    object 
 1   premium                          100 non-null    bool   
 2   name                             100 non-null    object 
 3   department                       0 non-null      float64
 4   has_test                         100 non-null    bool   
 5   response_letter_required         100 non-null    bool   
 6   address                          0 non-null      float64
 7   response_url                     0 non-null      object 
 8   sort_point_distance              0 non-null      object 
 9   published_at                     100 non-null    object 
 10  created_at                       100 non-null    object 
 11  archived                         100 non-null    bool   
 12  apply_alternate_url    

,department,address,insider_interview,salary.from,salary.to,salary_range.from,salary_range.to,employer.country_id,salary_range.frequency,address.lat,address.lng,address.description,address.metro.lat,address.metro.lng,branding.tariff,address.metro,employer.logo_urls
count,0.0,0.0,0.0,73.000000,76.000000,73.000000,76.000000,98.000000,0.0,48.000000,48.000000,0.0,35.000000,35.000000,0.0,0.0,0.0
mean,NaN,NaN,NaN,213512.328767,285365.789474,213512.328767,285365.789474,1.081633,NaN,55.022916,46.670872,NaN,56.480596,38.315910,NaN,NaN,NaN
std,NaN,NaN,NaN,147578.336020,148078.035781,147578.336020,148078.035781,0.586330,NaN,3.521817,28.234822,NaN,1.633762,9.206312,NaN,NaN,NaN
min,NaN,NaN,NaN,1500.000000,1900.000000,1500.000000,1900.000000,1.000000,NaN,45.013313,20.446392,NaN,55.040998,30.311293,NaN,NaN,NaN
25%,NaN,NaN,NaN,130000.000000,200000.000000,130000.000000,200000.000000,1.000000,NaN,55.452789,37.585726,NaN,55.725009,37.543750,NaN,NaN,NaN
50%,NaN,NaN,NaN,220000.000000,275000.000000,220000.000000,275000.000000,1.000000,NaN,55.744551,37.618409,NaN,55.752123,37.610388,NaN,NaN,NaN
75%,NaN,NaN,NaN,280000.000000,370000.000000,280000.000000,370000.000000,1.000000,NaN,55.783982,37.699714,NaN,55.821306,37.636285,NaN,NaN,NaN
max,NaN,NaN,NaN,850000.000000,950000.000000,850000.000000,950000.000000,6.000000,NaN,60.054554,142.729587,NaN,60.037098,82.917447,NaN,NaN,NaN


Очистка данных

78.4248
